# Price Data Exploration, Collection & Feature Engineering
## Crop Yield Risk & Advisory Dashboard

This notebook documents the **complete market-price pipeline** used in the Crop Yield Risk & Advisory Dashboard.

It is intentionally written as both:

- an executable project notebook,
- a methodology record,
- a debugging log,
- and a future viva/study reference.

The notebook records not only the final successful approach, but also important failed attempts, why they failed, and how the pipeline was corrected.

---

## Project Scope

### States
- Uttar Pradesh
- Punjab
- Haryana

### Crops
- Wheat
- Rice
- Maize

### Crop Years
- 2021-22
- 2022-23
- 2023-24
- 2024-25

### Price Source
**AGMARKNET 2.0**

### Final Price Grain

The final price table must contain one row for each:

`State + District + Crop + Year`

This key must remain unique before the price dataset is merged with production and rainfall.

---

## Main Price Features

The final annual price features include:

- `Avg_Modal_Price`
- `Min_Monthly_Price`
- `Max_Monthly_Price`
- `Price_Std`
- `Months_Available`
- `Avg_Markets_Reporting`
- `Price_Volatility_Pct`
- `Price_Coverage_Pct`
- `Price_Coverage_Quality`

> **Important rule:** missing price information is not zero price, and missing volatility is not zero volatility.


## How to Run This Notebook Safely

This notebook contains two kinds of material:

### Normal reproducible pipeline
These cells can be run when the project's processed/raw files are present.

### Historical / optional diagnostics
These cells document Selenium debugging, geocoding experiments, and Punjab
mapping recovery steps that were useful during development.

They are **not required** to rebuild the final master price table once the
final market mappings and annual state-crop feature files already exist.

To make `Run All` safer, optional collection/recovery code is controlled by
flags and skips itself by default.

```python
RUN_SELENIUM_DIAGNOSTICS = False
RUN_HISTORICAL_GEOCODING = False
RUN_HISTORICAL_MAPPING_RECOVERY = False
```

Set a flag to `True` only when you intentionally want to repeat that historical
step and have the required browser/intermediate files available.

## Required Libraries

The notebook uses a small set of core libraries throughout the price pipeline:

- `pandas` for loading, cleaning, grouping and merging tabular data.
- `numpy` for numerical operations and missing-value handling.
- `re` for filename parsing and market-name normalization.
- `time` for pauses while interacting with the dynamic AGMARKNET website.
- `calendar` for readable month names.
- `shutil` for moving downloaded CSV files.
- `pathlib.Path` for reliable file and folder handling.

Selenium and geocoding libraries are imported later only when needed.


In [1]:
# Core tabular data processing
import pandas as pd
import numpy as np

# Text cleaning and filename parsing
import re

# Timing and month utilities used by browser automation
import time
import calendar

# File movement and path handling
import shutil
from pathlib import Path


# ------------------------------------------------------------
# OPTIONAL/HISTORICAL EXECUTION FLAGS
# ------------------------------------------------------------
# Keep these False during a normal rerun of the completed project.
RUN_SELENIUM_DIAGNOSTICS = False
RUN_HISTORICAL_GEOCODING = False
RUN_HISTORICAL_MAPPING_RECOVERY = False


# ------------------------------------------------------------
# PROJECT PATH DISCOVERY
# ------------------------------------------------------------
# The notebook is normally stored under:
# project/analytics/notebooks/
#
# Instead of relying only on "../../", search upward for the project
# folder containing both "data" and "analytics".
current_dir = Path.cwd().resolve()

search_locations = [
    current_dir,
    *current_dir.parents
]

PROJECT_ROOT = None

for candidate in search_locations:
    if (
        (candidate / "data").exists()
        and (candidate / "analytics").exists()
    ):
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    # Fallback keeps the notebook usable for inspection even when it
    # was opened outside the original repository.
    PROJECT_ROOT = current_dir

    print(
        "WARNING: Project root could not be auto-detected."
    )
    print(
        "If you want to run file-processing cells, move this notebook "
        "back to the project repository or set PROJECT_ROOT manually."
    )

RAW_PRICE_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "prices"
)

PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

print("Working directory:", current_dir)
print("Project root:", PROJECT_ROOT)
print("Raw price folder:", RAW_PRICE_DIR)
print("Processed folder:", PROCESSED_DIR)

Working directory: E:\Crop-Yeild-Risk-Dashboard\analytics\notebooks
Project root: E:\Crop-Yeild-Risk-Dashboard
Raw price folder: E:\Crop-Yeild-Risk-Dashboard\data\raw\prices
Processed folder: E:\Crop-Yeild-Risk-Dashboard\data\processed


# 1. Project folders

This notebook assumes it is inside:

```text
analytics/notebooks/
```

Therefore we use:

```text
../../data/raw/prices
../../data/processed
```

Raw price files follow:

```text
{state}_{crop}_{year}_{month}.csv
```

Example:

```text
uttar_pradesh_wheat_2021_01.csv
punjab_wheat_2024_05.csv
```

# 2. AGMARKNET 2.0 source exploration

We navigated:

```text
AGMARKNET
→ Reports
→ Price Trend Reports
→ Market-wise Reports (All Districts)
→ Market-wise Wholesale Prices Monthly Analysis (All Districts)
```

Working filter page:

```text
https://agmarknet.gov.in/marketallwholesalemonthanalysis
```

The filters are cascading:

```text
Commodity → State/UT → Year → Month
```

The exported monthly report contains:
- Market
- selected month price
- previous month price
- same month previous year price
- percentage changes

For this project we intentionally keep **only the selected month's price**.

### Why?
If we retain previous-month and previous-year columns from every exported report, the same historical month can appear repeatedly and create duplicated observations.

# 3. First CSV-format issue

AGMARKNET exports are not normal CSV files.

Observed structure:

```text
row 1 = report title
row 2 = actual header
remaining rows = market observations + footer text
```

So the correct reader is:

```python
pd.read_csv(file_path, header=1)
```

and not the default `header=0`.

# 4. Failed attempt: treating every report row as a market

### What went wrong
The exported report contains footer/junk rows such as:
- `Average`
- `Note:`
- formula/explanation rows

Initially these could enter the dataset as if they were market names.

### Fix
Real market rows in these exports end with `APMC`, so we changed the cleaner to keep only:

```python
Market.str.lower().str.endswith("apmc")
```

This removed report-footer rows before mapping or aggregation.

In [2]:
def normalize_market_name(name):
    name = str(name).strip().lower()
    name = re.sub(r"\bapmc\b", "", name)
    name = re.sub(r"\s+", " ", name)
    return name.strip()


def clean_monthly_price_csv(file_path, state, crop, year, month):
    # Row 1 is report title; row 2 is the real header.
    df = pd.read_csv(file_path, header=1)

    # Second column = price for the selected month.
    current_price_column = df.columns[1]

    df = df[["Market", current_price_column]].copy()
    df = df.rename(columns={current_price_column: "Monthly_Price"})

    df["Market"] = df["Market"].astype(str).str.strip()

    # Remove Average / Note / formula rows.
    df = df[
        df["Market"].str.lower().str.endswith("apmc")
    ].copy()

    df["Monthly_Price"] = pd.to_numeric(
        df["Monthly_Price"],
        errors="coerce"
    )

    df["State"] = state
    df["Crop"] = crop
    df["Calendar_Year"] = year
    df["Month"] = month
    df["Month_Date"] = pd.Timestamp(
        year=year,
        month=month,
        day=1
    )

    return df[
        [
            "State",
            "Crop",
            "Calendar_Year",
            "Month",
            "Month_Date",
            "Market",
            "Monthly_Price"
        ]
    ]

# 5. Generic monthly-history loader

This function:
1. finds all files for one state-crop pair,
2. extracts year/month from filenames,
3. cleans each file,
4. concatenates them,
5. creates a normalized `Market_Key`.

In [3]:
def load_price_history(
    state,
    crop,
    raw_folder=None
):
    """Load every downloaded monthly CSV for one state-crop pair.

    Parameters
    ----------
    state : str
        Project state name, e.g. "Punjab".
    crop : str
        Project crop name, e.g. "Wheat".
    raw_folder : str or Path or None
        Optional override. When omitted, the notebook's detected
        RAW_PRICE_DIR is used.
    """

    if raw_folder is None:
        raw_folder = RAW_PRICE_DIR

    raw_folder = Path(raw_folder)

    state_slug = state.lower().replace(" ", "_")
    crop_slug = crop.lower().replace(" ", "_")

    files = sorted(
        raw_folder.glob(
            f"{state_slug}_{crop_slug}_*.csv"
        )
    )

    print(f"{state} | {crop}")
    print("Raw monthly files found:", len(files))

    if len(files) == 0:
        raise FileNotFoundError(
            f"No price files found for {state} - {crop} in "
            f"{raw_folder.resolve()}"
        )

    all_months = []

    for file_path in files:
        match = re.search(
            r"_(\d{4})_(\d{2})$",
            file_path.stem
        )

        if not match:
            print("Skipped unexpected filename:", file_path.name)
            continue

        year = int(match.group(1))
        month = int(match.group(2))

        all_months.append(
            clean_monthly_price_csv(
                file_path=file_path,
                state=state,
                crop=crop,
                year=year,
                month=month
            )
        )

    if not all_months:
        raise ValueError(
            "Files existed but no monthly tables were created."
        )

    monthly_df = pd.concat(
        all_months,
        ignore_index=True
    )

    monthly_df["Market_Key"] = (
        monthly_df["Market"]
        .apply(normalize_market_name)
    )

    print("Rows:", len(monthly_df))
    print("Months:", monthly_df["Month_Date"].nunique())
    print("Markets:", monthly_df["Market"].nunique())

    return monthly_df

# 6. Calendar month → crop year

The project uses crop years:

```text
July 2021 – June 2022 → 2021-22
July 2022 – June 2023 → 2022-23
July 2023 – June 2024 → 2023-24
July 2024 – June 2025 → 2024-25
```

January-June 2021 can exist as historical context, but those months are outside our final four crop years.

In [4]:
def get_crop_year(date):
    date = pd.to_datetime(date)

    if date.month >= 7:
        start_year = date.year
    else:
        start_year = date.year - 1

    return f"{start_year}-{(start_year + 1) % 100:02d}"

# 7. Failed/weak aggregation idea and the improved method

A district can have multiple markets.

### Weak approach
Directly calculate annual statistics from every market row.

### Why that is a problem
- districts with more reporting markets receive more weight,
- `Price_Std` can capture differences **between markets** rather than movement **over time**.

### Improved two-stage aggregation

**Stage 1:** for each district-month, calculate the **median** market price.

**Stage 2:** calculate annual features from those monthly district prices.

The median is used because it is less sensitive to extreme individual market observations.

In [5]:
def build_price_features(mapped_monthly_df):
    df = mapped_monthly_df.copy()

    df["Year"] = df["Month_Date"].apply(get_crop_year)

    target_years = [
        "2021-22",
        "2022-23",
        "2023-24",
        "2024-25"
    ]

    df = df[df["Year"].isin(target_years)].copy()

    # Stage 1: one district-month representative price.
    district_monthly = (
        df.groupby(
            [
                "State",
                "District",
                "Crop",
                "Year",
                "Month_Date"
            ],
            as_index=False
        )
        .agg(
            District_Monthly_Price=(
                "Monthly_Price",
                "median"
            ),
            Markets_Reporting=(
                "Monthly_Price",
                "count"
            )
        )
    )

    # Stage 2: annual features.
    features = (
        district_monthly
        .groupby(
            [
                "State",
                "District",
                "Crop",
                "Year"
            ],
            as_index=False
        )
        .agg(
            Avg_Modal_Price=(
                "District_Monthly_Price",
                "mean"
            ),
            Min_Monthly_Price=(
                "District_Monthly_Price",
                "min"
            ),
            Max_Monthly_Price=(
                "District_Monthly_Price",
                "max"
            ),
            Price_Std=(
                "District_Monthly_Price",
                "std"
            ),
            Months_Available=(
                "District_Monthly_Price",
                "count"
            ),
            Avg_Markets_Reporting=(
                "Markets_Reporting",
                "mean"
            )
        )
    )

    features["Price_Volatility_Pct"] = (
        features["Price_Std"]
        / features["Avg_Modal_Price"]
        * 100
    )

    features["Price_Coverage_Pct"] = (
        features["Months_Available"]
        / 12
        * 100
    )

    def coverage_quality(months):
        if months == 12:
            return "Complete"
        elif months >= 9:
            return "Good"
        elif months >= 6:
            return "Moderate"
        else:
            return "Low"

    features["Price_Coverage_Quality"] = (
        features["Months_Available"]
        .apply(coverage_quality)
    )

    # Fewer than 6 months is too little for reliable volatility.
    features.loc[
        features["Months_Available"] < 6,
        "Price_Volatility_Pct"
    ] = pd.NA

    return features

# 8. Why low-coverage volatility becomes NA

Formula:

```text
Price_Volatility_Pct = Price_Std / Avg_Modal_Price × 100
```

If only 1-5 monthly observations exist, the resulting volatility is not a reliable representation of the full crop year.

So:

```python
if Months_Available < 6:
    Price_Volatility_Pct = NA
```

We do **not** replace it with 0.

- `0` means prices were actually stable.
- `NA` means there was not enough evidence to estimate volatility.

# 9. Selenium setup

AGMARKNET 2.0 uses custom React dropdowns rather than simple HTML `<select>` elements.

Therefore we used Selenium to interact with visible dropdown options.

In [6]:
# Selenium is needed only for OPTIONAL live AGMARKNET collection and
# historical diagnostics. The completed price-feature pipeline can use
# already downloaded CSV files without opening a browser.

SELENIUM_AVAILABLE = False

try:
    from selenium import webdriver
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support.ui import WebDriverWait
    from selenium.webdriver.support import expected_conditions as EC
    from selenium.common.exceptions import (
        InvalidSessionIdException,
        WebDriverException,
        TimeoutException
    )

    SELENIUM_AVAILABLE = True
    print("Selenium available ✅")

except ImportError:
    print(
        "Selenium is not installed in this environment. "
        "Live AGMARKNET collection cells will be skipped."
    )


def create_new_driver():
    """Create a Chrome driver only when Selenium is available."""

    if not SELENIUM_AVAILABLE:
        raise RuntimeError(
            "Selenium is unavailable. Install Selenium/ChromeDriver "
            "only if you intentionally want to repeat live collection."
        )

    driver = webdriver.Chrome()
    time.sleep(2)
    return driver


def select_exact_dropdown_option(
    driver,
    label_text,
    option_text
):
    """Select one exact option from an AGMARKNET React dropdown.

    The helper waits until cascading controls become enabled.
    """

    if not SELENIUM_AVAILABLE:
        raise RuntimeError(
            "Selenium is unavailable."
        )

    WebDriverWait(
        driver,
        30
    ).until(
        EC.presence_of_element_located(
            (
                By.XPATH,
                f"//label[normalize-space()='{label_text}']"
            )
        )
    )

    def get_enabled_trigger(d):
        current_label = d.find_element(
            By.XPATH,
            f"//label[normalize-space()='{label_text}']"
        )

        current_trigger = current_label.find_element(
            By.XPATH,
            "./preceding-sibling::div[1]"
        )

        if current_trigger.get_attribute("tabindex") != "-1":
            return current_trigger

        return False

    trigger = WebDriverWait(
        driver,
        30
    ).until(
        get_enabled_trigger
    )

    time.sleep(1.5)

    driver.execute_script(
        "arguments[0].click();",
        trigger
    )

    time.sleep(1)

    option_xpath = (
        "//div[contains(@class,'cursor-pointer')]"
        f"[.//span[normalize-space()='{option_text}']]"
    )

    visible_options = WebDriverWait(
        driver,
        30
    ).until(
        lambda d: [
            element
            for element in d.find_elements(
                By.XPATH,
                option_xpath
            )
            if element.is_displayed()
        ]
    )

    driver.execute_script(
        "arguments[0].click();",
        visible_options[0]
    )

    time.sleep(2)

    print(
        f"{label_text} → {option_text} selected"
    )

Selenium available ✅


# 10. Uttar Pradesh — market mapping process

The price data is reported by **market**, but production/rainfall are at **district** level.

So we needed:

```text
Market → District
```

### Attempt 1: official mapping references
This mapped many markets, but:
- some markets were missing,
- some names had conflicting district references,
- spellings differed.

We accepted a mapping automatically only when:

```text
one normalized Market_Key → exactly one district
```

### Attempt 2: geocoding leftovers
Remaining markets were geocoded and compared with canonical district names from the production data.

### Final manual review
Only unresolved or naming-mismatch markets were corrected manually.

# 11. Final manually reviewed Uttar Pradesh mappings

```python
{
 "ahirora":"Mirzapur",
 "anoop shahar":"Bulandshahr",
 "bachranwa":"Rae Bareli",
 "badda":"Shahjahanpur",
 "billsadda":"Pilibhit",
 "dibiapur":"Auraiya",
 "jamanian":"Ghazipur",
 "jasvantnagar":"Etawah",
 "jhansi grain":"Jhansi",
 "lediyari":"Prayagraj",
 "rampurmaniharan":"Saharanpur",
 "samsabad":"Agra",
 "sikanderabad":"Bulandshahr",
 "vilthararoad":"Ballia",
 "gopiganj":"Bhadohi",
 "mohammdi":"Kheri"
}
```

Two key canonical-name decisions:
- geocoder `Sant Ravidas Nagar` → production district `Bhadohi`
- geocoder `Lakhimpur Kheri` → production district `Kheri`

In [7]:
final_manual_mapping = {
    "ahirora": "Mirzapur",
    "anoop shahar": "Bulandshahr",
    "bachranwa": "Rae Bareli",
    "badda": "Shahjahanpur",
    "billsadda": "Pilibhit",
    "dibiapur": "Auraiya",
    "jamanian": "Ghazipur",
    "jasvantnagar": "Etawah",
    "jhansi grain": "Jhansi",
    "lediyari": "Prayagraj",
    "rampurmaniharan": "Saharanpur",
    "samsabad": "Agra",
    "sikanderabad": "Bulandshahr",
    "vilthararoad": "Ballia",
    "gopiganj": "Bhadohi",
    "mohammdi": "Kheri"
}

# 12. Uttar Pradesh Wheat — verified results

After the cleaner was corrected:

```text
54 monthly files
231 unique markets
about 10,503 cleaned market-month rows
```

Final price feature table:

```text
302 district-crop-year rows
```

Observed coverage:

```text
12 months → 279 rows
11 months → 4
10 months → 3
9 months  → 3
8 months  → 3
6 months  → 1
5 months  → 1
2 months  → 2
1 month   → 4
0 months  → 2
```

Final files:

```text
up_market_district_mapping.csv
up_wheat_monthly_prices_mapped_2021_2025.csv
up_wheat_price_features_2021_2025.csv
```

# 13. Uttar Pradesh Maize — verified results

The already validated UP market mapping was reused.

Observed:

```text
54 monthly raw files
0 unmapped markets
163 feature rows
0 duplicate district-crop-year rows
41 missing volatility values
```

Coverage:

```text
Complete → 90
Good     → 19
Moderate → 13
Low      → 41
```

Saved:

```text
up_maize_monthly_prices_mapped_2021_2025.csv
up_maize_price_features_2021_2025.csv
```

# 14. Uttar Pradesh Rice — commodity problem

AGMARKNET showed several related commodities:

```text
Basmati Rice
Beaten Rice
Broken Rice
Chakhao (Black Rice)
Paddy (Basmati)
Paddy (Common)
Rice
Rice Bean
```

### Decision
For project crop `Rice`, use AGMARKNET:

```text
Paddy(Common)
```

### Why?
Our production dataset represents the crop/farmer stage. Common paddy is closer to that stage than processed rice, while Basmati is a specialty segment.

This assumption must be documented in the final methodology.

# 15. Rice alias solution

AGMARKNET needed the filter name:

```text
Paddy(Common)
```

but our project files needed:

```text
rice
```

So we collected using `Paddy(Common)` and renamed the downloaded file to the project crop alias.

Example:

```text
uttar_pradesh_paddy(common)_2021_01.csv
        ↓
uttar_pradesh_rice_2021_01.csv
```

# 16. Uttar Pradesh Rice — verified results

Observed monthly data:

```text
54 months
165 markets
4,724 rows
0 unmapped markets after applying UP mapping
```

Feature results:

```text
263 rows
0 duplicate district-crop-year rows
126 missing volatility values
```

Coverage:

```text
Complete → 65
Good     → 29
Moderate → 43
Low      → 126
```

Saved:

```text
up_rice_monthly_prices_mapped_2021_2025.csv
up_rice_price_features_2021_2025.csv
```

# 17. Punjab Wheat — first approach failed

We initially reused the assumption:

```text
every state-crop-year has all 12 months
```

For:

```text
Punjab + Wheat + 2021 + January
```

Commodity, State and Year selections worked, but selecting Month caused:

```text
TimeoutException
```

The failure occurred while trying to select `January`.

Instead of assuming Selenium was broken, we inspected the Month dropdown.

# 18. Punjab month diagnostic

Observed:

```text
Month tabindex = 0
```

This meant the Month control was enabled.

But the visible month-name search returned:

```text
January   → 0
February  → 0
March     → 0
April     → 2
May       → 2
June      → 0
July      → 0
August    → 0
September → 0
October   → 0
November  → 0
December  → 0
```

### Conclusion
The timeout was **not** simply a Selenium click failure.

AGMARKNET was only exposing months for which data existed for that combination.

In [8]:
# OPTIONAL HISTORICAL DIAGNOSTIC
#
# This cell was originally used to investigate why January 2021 was
# not offered for Punjab Wheat. It is not needed for the final price
# feature pipeline.

if not RUN_SELENIUM_DIAGNOSTICS:
    print(
        "Skipped historical Punjab Month-dropdown diagnostic "
        "(RUN_SELENIUM_DIAGNOSTICS=False)."
    )

elif not SELENIUM_AVAILABLE:
    print(
        "Skipped: Selenium is not available."
    )

else:
    # Create a live browser only when the user explicitly requests the
    # historical diagnostic.
    driver = create_new_driver()

    try:
        # Reproduce the source combination that originally failed.
        driver.get(
            "https://agmarknet.gov.in/"
            "marketallwholesalemonthanalysis"
        )
        time.sleep(4)

        select_exact_dropdown_option(
            driver,
            "Select Commodity",
            "Wheat"
        )

        select_exact_dropdown_option(
            driver,
            "State/UT",
            "Punjab"
        )

        select_exact_dropdown_option(
            driver,
            "Year",
            "2021"
        )

        month_label = driver.find_element(
            By.XPATH,
            "//label[normalize-space()='Month']"
        )

        month_trigger = month_label.find_element(
            By.XPATH,
            "./preceding-sibling::div[1]"
        )

        print(
            "Month tabindex:",
            month_trigger.get_attribute("tabindex")
        )

        driver.execute_script(
            "arguments[0].click();",
            month_trigger
        )

        time.sleep(2)

        month_names = [
            "January",
            "February",
            "March",
            "April",
            "May",
            "June",
            "July",
            "August",
            "September",
            "October",
            "November",
            "December"
        ]

        for month in month_names:
            matches = driver.find_elements(
                By.XPATH,
                f"//*[normalize-space()='{month}']"
            )

            visible = [
                element
                for element in matches
                if element.is_displayed()
            ]

            print(
                month,
                "→",
                len(visible)
            )

    finally:
        driver.quit()

Skipped historical Punjab Month-dropdown diagnostic (RUN_SELENIUM_DIAGNOSTICS=False).


# 19. Fix: discover available months before downloading

New method:

```text
for each year
    ↓
open AGMARKNET
    ↓
select Commodity + State + Year
    ↓
inspect which Month options exist
    ↓
download only those months
    ↓
record other months as unavailable
```

This cleanly separates:

- **UNAVAILABLE** → source does not offer the month
- **FAILED** → month should exist but technical download failed
- **SKIPPED** → local file already exists
- **SUCCESS** → downloaded now

This is much better than pretending every missing source month is an automation error.

In [9]:
def get_available_months(
    driver,
    state,
    crop,
    year
):
    """Return only the months AGMARKNET currently exposes.

    This helper belongs to the OPTIONAL live-collection workflow.
    """

    if not SELENIUM_AVAILABLE:
        raise RuntimeError(
            "Selenium is unavailable."
        )

    filter_url = (
        "https://agmarknet.gov.in/"
        "marketallwholesalemonthanalysis"
    )

    driver.get(filter_url)
    time.sleep(4)

    select_exact_dropdown_option(
        driver,
        "Select Commodity",
        crop
    )

    select_exact_dropdown_option(
        driver,
        "State/UT",
        state
    )

    select_exact_dropdown_option(
        driver,
        "Year",
        str(year)
    )

    month_label = WebDriverWait(
        driver,
        30
    ).until(
        EC.presence_of_element_located(
            (
                By.XPATH,
                "//label[normalize-space()='Month']"
            )
        )
    )

    month_trigger = month_label.find_element(
        By.XPATH,
        "./preceding-sibling::div[1]"
    )

    driver.execute_script(
        "arguments[0].click();",
        month_trigger
    )

    time.sleep(2)

    month_names = [
        "January",
        "February",
        "March",
        "April",
        "May",
        "June",
        "July",
        "August",
        "September",
        "October",
        "November",
        "December"
    ]

    available = []

    for month_number, month_name in enumerate(
        month_names,
        start=1
    ):
        matches = driver.find_elements(
            By.XPATH,
            f"//*[normalize-space()='{month_name}']"
        )

        visible_matches = [
            element
            for element in matches
            if element.is_displayed()
        ]

        if visible_matches:
            available.append(
                {
                    "Month": month_number,
                    "Month_Name": month_name
                }
            )

    return pd.DataFrame(available)

# 20. Punjab Wheat — verified collection result

After switching to available-month discovery:

```text
Months actually available: 30
First month: 2021-04-01
Last month: 2025-06-01
Unique markets: 178
Total cleaned rows: 1,582
```

Important:

> Missing AGMARKNET months are kept as missing source coverage. They are never converted to price = 0.

# 21. Punjab official market-mapping attempt

We correctly started a **new Punjab mapping** instead of reusing UP.

Observed:

```text
178 unique Punjab Wheat markets
38 safe official market mappings
1,339 rows still missing District
151 unique markets still unmapped
```

The official reference was therefore not sufficient on its own.

# 22. Punjab geocoding attempt

Instead of manually mapping all 151 remaining markets, we geocoded them.

Method:

```text
Market
→ remove APMC
→ query "<market>, Punjab, India"
→ retrieve candidate district
→ compare with canonical production districts
```

Observed:

```text
Total markets: 151
Locations found: 85
Locations not found: 66
Candidate districts available: 85

Exact production-district matches: 77
District names needing review: 8
```

Only the **77 exact matches** were accepted automatically.

In [10]:
# OPTIONAL HISTORICAL GEOCODING SETUP
#
# The final Punjab mapping has already been completed and saved.
# This helper is retained to document how unresolved markets were
# originally investigated.

GEOPY_AVAILABLE = False
geocode_market = None

try:
    from geopy.geocoders import Nominatim
    from geopy.extra.rate_limiter import RateLimiter
    GEOPY_AVAILABLE = True
except ImportError:
    print(
        "geopy is not installed. Historical geocoding will be skipped."
    )


if RUN_HISTORICAL_GEOCODING and GEOPY_AVAILABLE:
    geolocator = Nominatim(
        user_agent=(
            "crop-yield-risk-dashboard-"
            "punjab-market-mapping"
        ),
        timeout=10
    )

    geocode_market = RateLimiter(
        geolocator.geocode,
        min_delay_seconds=1.2,
        max_retries=2,
        error_wait_seconds=5,
        swallow_exceptions=True
    )


def get_punjab_market_candidate(market_name):
    """Historical helper used to geocode an unresolved Punjab market.

    It runs only when RUN_HISTORICAL_GEOCODING=True.
    """

    if not RUN_HISTORICAL_GEOCODING:
        raise RuntimeError(
            "Historical geocoding is disabled. "
            "Set RUN_HISTORICAL_GEOCODING=True only if you "
            "intentionally want to repeat geocoding."
        )

    if not GEOPY_AVAILABLE or geocode_market is None:
        raise RuntimeError(
            "geopy/geocoder is unavailable."
        )

    clean_name = (
        str(market_name)
        .replace("APMC", "")
        .strip()
    )

    query = f"{clean_name}, Punjab, India"

    location = geocode_market(
        query,
        addressdetails=True
    )

    if location is None:
        return {
            "Geocode_Found": False,
            "Candidate_District": None,
            "Candidate_Place": None,
            "Full_Address": None,
            "Latitude": None,
            "Longitude": None
        }

    address = location.raw.get(
        "address",
        {}
    )

    candidate_district = (
        address.get("state_district")
        or address.get("county")
        or address.get("district")
        or address.get("city_district")
    )

    candidate_place = (
        address.get("city")
        or address.get("town")
        or address.get("village")
        or address.get("municipality")
        or address.get("suburb")
    )

    return {
        "Geocode_Found": True,
        "Candidate_District": candidate_district,
        "Candidate_Place": candidate_place,
        "Full_Address": location.address,
        "Latitude": location.latitude,
        "Longitude": location.longitude
    }

# 23. Important engineering step: save geocoding immediately

Geocoding 151 markets is slow because requests are rate-limited.

So we saved:

```text
../../data/processed/
punjab_unmapped_market_geocode_candidates.csv
```

This later became very useful when notebook variables disappeared.

# 24. Failure: `punjab_geocoded` was not defined

Later the notebook said the `punjab_geocoded` object did not exist.

### Cause
The Jupyter kernel/session had lost the variable.

### Bad solution
Geocode all 151 markets again.

### Correct solution
Reload the saved CSV and recreate only the derived variables.

> **Run-All note:** The recovery code below is now optional and skipped
> by default. The temporary geocode-candidate CSV is not required once
> `punjab_market_district_mapping.csv` exists.


In [11]:
# OPTIONAL HISTORICAL RECOVERY CELL
#
# This CSV was an intermediate recovery artifact. It is NOT required
# by the final price pipeline. The final Punjab market mapping is the
# important reusable output.

punjab_geocode_file = (
    PROCESSED_DIR
    / "punjab_unmapped_market_geocode_candidates.csv"
)

if not RUN_HISTORICAL_MAPPING_RECOVERY:
    print(
        "Skipped historical Punjab geocode-recovery cell "
        "(RUN_HISTORICAL_MAPPING_RECOVERY=False)."
    )

elif not punjab_geocode_file.exists():
    print(
        "Historical intermediate file not found:"
    )
    print(
        punjab_geocode_file.resolve()
    )
    print(
        "This is OK for the completed project. "
        "Use punjab_market_district_mapping.csv instead."
    )

else:
    punjab_geocoded = pd.read_csv(
        punjab_geocode_file
    )

    production_file = (
        PROCESSED_DIR
        / "production_history_2021_2025.csv"
    )

    if not production_file.exists():
        print(
            "Cannot continue historical recovery because "
            "production_history_2021_2025.csv is missing."
        )
    else:
        production_history = pd.read_csv(
            production_file
        )

        production_punjab_districts = sorted(
            production_history.loc[
                production_history["State"]
                .astype(str)
                .str.strip()
                .str.lower()
                == "punjab",
                "District"
            ]
            .dropna()
            .unique()
        )

        punjab_geocoded_found = (
            punjab_geocoded[
                punjab_geocoded[
                    "Candidate_District"
                ].notna()
            ]
            .copy()
        )

        punjab_geocoded_found[
            "Exact_Production_Match"
        ] = (
            punjab_geocoded_found[
                "Candidate_District"
            ].isin(
                production_punjab_districts
            )
        )

        print(
            "Locations found:",
            len(punjab_geocoded_found)
        )

        print(
            "Exact matches:",
            punjab_geocoded_found[
                "Exact_Production_Match"
            ].sum()
        )

        print(
            "Name mismatches:",
            (
                ~punjab_geocoded_found[
                    "Exact_Production_Match"
                ]
            ).sum()
        )

Skipped historical Punjab geocode-recovery cell (RUN_HISTORICAL_MAPPING_RECOVERY=False).


Expected recovery values from the previous successful run:

```text
Locations found: 85
Exact matches: 77
Name mismatches: 8
```

# 25. Failure: `No objects to concatenate`

While recovering the official Punjab mapping, this appeared:

```text
ValueError: No objects to concatenate
```

### Cause

The code had:

```python
punjab_mapping_parts = []
```

but no valid Punjab tables were appended before:

```python
pd.concat(punjab_mapping_parts)
```

### Fix

Always guard concatenation:

```python
if len(punjab_mapping_parts) == 0:
    ...
else:
    pd.concat(...)
```

Possible reasons the list can be empty:
- the original reference CSV is no longer in the searched folder,
- a CSV does not contain `state`, `district`, and `market`,
- the candidate reference contains no Punjab rows.

In [12]:
# OPTIONAL HISTORICAL SAFE-CONCAT EXAMPLE
#
# `punjab_mapping_parts` existed only during the original mapping
# reconstruction. Normal reruns do not need this variable.

if not RUN_HISTORICAL_MAPPING_RECOVERY:
    print(
        "Skipped historical official-mapping recovery "
        "(RUN_HISTORICAL_MAPPING_RECOVERY=False)."
    )

elif "punjab_mapping_parts" not in globals():
    print(
        "punjab_mapping_parts is not present in this session. "
        "Nothing to recover."
    )

elif len(punjab_mapping_parts) == 0:
    print(
        "No Punjab official reference rows found."
    )

    punjab_safe_mapping = pd.DataFrame(
        columns=[
            "Market_Key",
            "District"
        ]
    )

else:
    punjab_official_mapping = pd.concat(
        punjab_mapping_parts,
        ignore_index=True
    )

    print(
        "Recovered Punjab official mapping rows:",
        len(punjab_official_mapping)
    )

Skipped historical official-mapping recovery (RUN_HISTORICAL_MAPPING_RECOVERY=False).


# 26. Failure: `load_price_history is not defined`

Later another error appeared:

```text
NameError: load_price_history is not defined
```

Again, the raw price files were **not lost**.

Only the helper-function definition had disappeared from the active kernel.

### Fix
Re-run/recover:
- `normalize_market_name`
- `clean_monthly_price_csv`
- `load_price_history`

Then reload:

```python
punjab_wheat_monthly = load_price_history(
    state="Punjab",
    crop="Wheat"
)
```

Expected previously verified values:

```text
Months: 30
Markets: 178
Rows: 1,582
```

# 27. Historical Punjab Recovery Checkpoint

Everything below has already been verified:

```text
Punjab Wheat raw/cleaned data
-----------------------------
Months available: 30
First month:      April 2021
Last month:       June 2025
Markets:          178
Rows:             1,582

Official mapping
----------------
Safe mappings: 38

Geocoding
---------
Markets sent:                       151
Locations found:                     85
Locations not found:                 66
Exact production-district matches:   77
District-name mismatches:              8
```

The Punjab mapping is **not yet complete**.

The work was interrupted during recovery/combination because Jupyter had forgotten helper variables/functions.

> **Historical note:** This section records the state of the work during an earlier recovery point. Punjab processing was later completed for Wheat, Maize and Rice. The completed results are documented in the continuation sections near the end of this notebook.


# 28. Historical Continuation Plan Used During Punjab Recovery

Once `punjab_safe_mapping`, `punjab_geocoded_found`, and `load_price_history` are available, combine:

```text
38 official-safe mappings
+
77 exact geocoder mappings
```

Only geocoder rows whose district is an **exact production-district match** should be added automatically.

> This was the planned recovery step at that time. It is retained because it explains how the final Punjab mapping was reconstructed after Jupyter state loss.


In [13]:
# OPTIONAL HISTORICAL MAPPING-COMBINATION CELL
#
# This block documents how the recovery objects were combined.
# It is skipped during normal reruns because the final reusable mapping
# has already been saved to disk.

required_recovery_objects = [
    "punjab_geocoded_found",
    "punjab_safe_mapping"
]

missing_recovery_objects = [
    name
    for name in required_recovery_objects
    if name not in globals()
]

if not RUN_HISTORICAL_MAPPING_RECOVERY:
    print(
        "Skipped historical Punjab mapping combination "
        "(RUN_HISTORICAL_MAPPING_RECOVERY=False)."
    )

elif missing_recovery_objects:
    print(
        "Cannot run historical combination. Missing objects:",
        missing_recovery_objects
    )

else:
    punjab_exact_geocode_mapping = (
        punjab_geocoded_found[
            punjab_geocoded_found[
                "Exact_Production_Match"
            ]
        ][
            [
                "Market_Key",
                "Candidate_District"
            ]
        ]
        .rename(
            columns={
                "Candidate_District": "District"
            }
        )
        .drop_duplicates(
            subset=["Market_Key"]
        )
    )

    existing_keys = set(
        punjab_safe_mapping[
            "Market_Key"
        ]
    )

    punjab_new_exact_mapping = (
        punjab_exact_geocode_mapping[
            ~punjab_exact_geocode_mapping[
                "Market_Key"
            ].isin(existing_keys)
        ]
        .copy()
    )

    punjab_market_mapping_v2 = pd.concat(
        [
            punjab_safe_mapping[
                ["Market_Key", "District"]
            ],
            punjab_new_exact_mapping[
                ["Market_Key", "District"]
            ]
        ],
        ignore_index=True
    )

    punjab_market_mapping_v2 = (
        punjab_market_mapping_v2
        .drop_duplicates(
            subset=["Market_Key"],
            keep="first"
        )
    )

    print(
        "Recovered mapped markets:",
        punjab_market_mapping_v2[
            "Market_Key"
        ].nunique()
    )

Skipped historical Punjab mapping combination (RUN_HISTORICAL_MAPPING_RECOVERY=False).


# 29. Load the Final Punjab Mapping

The earlier development version depended on temporary recovery objects
such as `punjab_market_mapping_v2`.

The corrected notebook now loads the final reusable file directly:

`punjab_market_district_mapping.csv`

If the raw Punjab Wheat monthly files are still present, the code also
demonstrates remapping them. If they are absent, the cell prints a
message and continues instead of failing.

In [14]:
# FINAL/REUSABLE PUNJAB MAPPING CHECK
#
# Use the completed mapping saved to disk. This avoids depending on
# temporary historical geocoding variables.

final_punjab_mapping_file = (
    PROCESSED_DIR
    / "punjab_market_district_mapping.csv"
)

if not final_punjab_mapping_file.exists():
    print(
        "Final Punjab mapping not found:"
    )
    print(
        final_punjab_mapping_file.resolve()
    )
    print(
        "This notebook can still be read as documentation, "
        "but Punjab remapping cannot be rerun until the file is restored."
    )

else:
    punjab_market_mapping = pd.read_csv(
        final_punjab_mapping_file
    )

    print(
        "Final Punjab market mapping loaded ✅"
    )
    print(
        "Mapping rows:",
        len(punjab_market_mapping)
    )

    # Rebuild the mapped Wheat history only when the raw monthly files
    # are present. Otherwise simply retain the completed mapping.
    try:
        punjab_wheat_monthly = load_price_history(
            state="Punjab",
            crop="Wheat"
        )

        punjab_wheat_with_district = (
            punjab_wheat_monthly
            .drop(
                columns=["District"],
                errors="ignore"
            )
            .merge(
                punjab_market_mapping[
                    ["Market_Key", "District"]
                ],
                on="Market_Key",
                how="left",
                validate="many_to_one"
            )
        )

        print(
            "Rows missing District:",
            punjab_wheat_with_district[
                "District"
            ].isna().sum()
        )

        print(
            "Unmapped markets:",
            punjab_wheat_with_district.loc[
                punjab_wheat_with_district[
                    "District"
                ].isna(),
                "Market"
            ].nunique()
        )

    except FileNotFoundError as error:
        print(
            "Raw Punjab Wheat monthly files are not present, "
            "so the mapped monthly table was not rebuilt."
        )
        print(error)

Final Punjab market mapping loaded ✅
Mapping rows: 196
Punjab | Wheat
Raw monthly files found: 30
Rows: 1582
Months: 30
Markets: 178
Rows missing District: 0
Unmapped markets: 0


# 30. Review the 8 district-name mismatches

Do not automatically force these names.

Print them beside the production district names and review each case.

In [15]:
# OPTIONAL HISTORICAL REVIEW
#
# The eight district-name mismatches were reviewed during development.
# The final mapping is already stored in
# punjab_market_district_mapping.csv.

if not RUN_HISTORICAL_MAPPING_RECOVERY:
    print(
        "Skipped historical Punjab mismatch review "
        "(RUN_HISTORICAL_MAPPING_RECOVERY=False)."
    )

elif (
    "punjab_geocoded_found" not in globals()
    or "production_punjab_districts" not in globals()
):
    print(
        "Historical recovery objects are unavailable. "
        "Nothing to review."
    )

else:
    punjab_name_mismatches = (
        punjab_geocoded_found[
            ~punjab_geocoded_found[
                "Exact_Production_Match"
            ]
        ][
            [
                "Market",
                "Market_Key",
                "Candidate_District",
                "Candidate_Place",
                "Full_Address"
            ]
        ]
        .drop_duplicates()
    )

    print(
        punjab_name_mismatches.to_string(
            index=False
        )
    )

    print(
        "\nCanonical Punjab production districts:"
    )

    print(
        production_punjab_districts
    )

Skipped historical Punjab mismatch review (RUN_HISTORICAL_MAPPING_RECOVERY=False).


# 31. Validation checklist for every price feature file

Before saving a state-crop feature table, verify:

### Key uniqueness
Only one row per:

```text
State + District + Crop + Year
```

### Coverage
Inspect:
- `Months_Available`
- `Price_Coverage_Quality`
- missing `Price_Volatility_Pct`

### Price sanity
Inspect:
- negative values,
- zero values,
- extreme outliers.

### Mapping
Review remaining missing districts before feature generation.

### Imputation
Do **not** blindly impute price features yet.

Imputation decisions should be made after all states/crops are combined and the overall missingness pattern is visible.

In [16]:
def validate_price_features(features):
    duplicates = features.duplicated(
        subset=[
            "State",
            "District",
            "Crop",
            "Year"
        ]
    ).sum()

    print(
        "Duplicate district-crop-year rows:",
        duplicates
    )

    print("\nCoverage quality:")
    print(
        features[
            "Price_Coverage_Quality"
        ].value_counts(
            dropna=False
        )
    )

    print("\nMissing volatility:")
    print(
        features[
            "Price_Volatility_Pct"
        ].isna().sum()
    )

    print("\nMinimum average price:")
    print(
        features[
            "Avg_Modal_Price"
        ].min()
    )

# 32. Complete methodology flow

```text
AGMARKNET monthly report
        ↓
Selenium collection
        ↓
Save each available month
        ↓
Read CSV using header=1
        ↓
Keep Market + current selected month price
        ↓
Remove non-APMC footer rows
        ↓
Normalize Market_Key
        ↓
Map Market → District
        ↓
Use safe official mappings
        ↓
Geocode unresolved markets
        ↓
Accept exact production-district matches
        ↓
Manually inspect naming mismatches / non-geocoded markets
        ↓
Convert month → crop year
        ↓
Median across markets within district-month
        ↓
Aggregate monthly district prices to annual features
        ↓
Calculate volatility and coverage
        ↓
Set volatility = NA when Months_Available < 6
        ↓
Validate uniqueness and missingness
```

# 33. Failure log — concise revision

### Failure 1: footer rows treated as markets
**Fix:** keep only market names ending in `APMC`.

### Failure 2: direct market-row annual aggregation could bias districts
**Fix:** first calculate district-month median, then annual features.

### Failure 3: no simple Rice commodity label
**Fix:** use `Paddy(Common)` as documented project proxy for Rice.

### Failure 4: Punjab January 2021 Month selector timed out
**Investigation:** dropdown was enabled, but January was not offered.
**Fix:** discover available months first.

### Failure 5: `punjab_geocoded` disappeared
**Cause:** notebook state lost.
**Fix:** reload saved geocoding CSV.

### Failure 6: `No objects to concatenate`
**Cause:** empty `punjab_mapping_parts`.
**Fix:** test list length before `pd.concat`.

### Failure 7: `load_price_history is not defined`
**Cause:** helper-function definition lost from kernel.
**Fix:** restore helper functions and reload existing raw CSV files.

### Main engineering lesson
Important intermediate results must be saved to disk.  
Do not make the entire pipeline depend on temporary notebook variables.

# 34. Files produced so far

### Uttar Pradesh

```text
up_market_district_mapping.csv
up_wheat_monthly_prices_mapped_2021_2025.csv
up_wheat_price_features_2021_2025.csv
up_maize_monthly_prices_mapped_2021_2025.csv
up_maize_price_features_2021_2025.csv
up_rice_monthly_prices_mapped_2021_2025.csv
up_rice_price_features_2021_2025.csv
```

### Punjab

```text
punjab_unmapped_market_geocode_candidates.csv
```

Punjab Wheat final mapping/features are still in progress.

# 35. Historical Checkpoint — Superseded

The original version of this notebook ended here while Punjab mapping recovery was still in progress.

That checkpoint is preserved as part of the development history, but it is no longer the final project status.

The continuation below documents the work completed afterwards:

- final Punjab Wheat mapping/features,
- Punjab Maize,
- Punjab Rice,
- Haryana official market reference,
- Haryana Wheat,
- Haryana Maize,
- Haryana Rice,
- all nine state-crop feature tables,
- the combined master price dataset,
- removal of zero-month placeholder rows,
- and final validation before Production + Rainfall + Price integration.


# 36. Final Working AGMARKNET Export Strategy

The earlier notebook documented month discovery, but the later Haryana work revealed two additional Selenium issues that are important to preserve.

## Exact Export Selector

During the first Haryana Wheat run:

- 38 months were discovered as available,
- 16 months were genuinely unavailable,
- but all 38 expected downloads failed.

The report itself was loading correctly.

The problem was an overly broad XPath for `Export as CSV`. It matched multiple ancestor elements instead of the actual clickable CSV item.

The corrected logic uses exact text:

`//*[normalize-space(text())='Export as CSV']`

and then finds the closest clickable parent.

## JavaScript String Failure

A later version passed the clickable-parent CSS selector as an invalid multi-line JavaScript string and Selenium returned:

`javascript error: Invalid or unexpected token`

The fix was to keep the JavaScript `closest(...)` call as one valid expression.

These failures were browser-automation problems, not missing AGMARKNET data.


In [17]:
# Final selector pattern used for AGMARKNET CSV export.

# Locate the Export button itself.
export_xpath = (
    "//button[normalize-space()='Export' "
    "or .//*[normalize-space()='Export']]"
)

# Locate only the exact CSV menu text.
csv_xpath = "//*[normalize-space(text())='Export as CSV']"

# After locating the exact text element as `csv_text`, find its nearest
# clickable parent. Keep the JavaScript expression on one line.
#
# csv_clickable = driver.execute_script(
#     'return arguments[0].closest(\'button, a, [role="menuitem"], '
#     '[role="option"], .cursor-pointer\');',
#     csv_text
# )
#
# The commented form above is kept as documentation because `csv_text`
# exists only inside the monthly-download helper.


# 37. Final Adaptive Collection Logic

The final collector became restartable and source-aware.

For each calendar year it:

1. opens AGMARKNET,
2. selects Commodity, State and Year,
3. discovers which Month options actually exist,
4. records unavailable months separately,
5. skips files already downloaded,
6. retries technical download failures,
7. recreates the Chrome driver if the Selenium session dies.

For calendar year 2025, only January through June are needed because July 2025 begins crop year `2025-26`, which is outside the project scope.

The collector distinguishes:

- **SUCCESS** — downloaded now,
- **SKIPPED** — already exists locally,
- **UNAVAILABLE** — source does not expose that month,
- **FAILED** — technical collection failure.

This distinction prevents genuine source gaps from being confused with scraper errors.


# 38. Final Rice Alias Rule

The project crop is called `Rice`, but AGMARKNET collection uses the commodity:

`Paddy(Common)`

Therefore:

- website filter name = `Paddy(Common)`
- project crop name = `Rice`
- saved filename uses `rice`

Example:

`Paddy(Common)` on AGMARKNET  
→ `haryana_rice_2024_10.csv`

This alias is used consistently for Uttar Pradesh, Punjab and Haryana.


# 39. Punjab Wheat — Final Completed Results

Punjab Wheat was completed after the recovery steps documented earlier.

Verified collection:

- **30 available months**
- **178 markets**
- **1,582 cleaned market-month rows**

Final annual price features:

- **87 feature rows**
- **0 duplicate State-District-Crop-Year keys**

Coverage:

- Low: 85
- Moderate: 2

Production comparison:

- 92 production keys
- 87 keys with price features
- 5 production keys without price features

Missing source coverage was retained as missing and was never converted to price = 0.


# 40. Punjab Maize — Final Completed Results

The completed Punjab master market mapping was reused for Maize.

One additional market mapping was required:

`Mansura` → `Ludhiana`

Final annual feature table:

- **57 price feature rows**

Coverage:

- Low: 53
- Moderate: 3
- Good: 1

Production comparison:

- 79 production keys
- 48 with price data
- 31 without price data

There were also 9 price-feature rows that did not correspond to production keys. These were retained during price exploration rather than silently discarded.


# 41. Punjab Rice — Final Completed Results

Punjab Rice was collected using the `Paddy(Common)` → `Rice` alias.

Verified collection:

- **17 available months**
- **164 markets**
- **1,200 cleaned rows**
- first month: January 2021
- last month: December 2024

Additional market mappings resolved during Rice processing included:

- Goniana → Bathinda
- Narot J Singh → Pathankot
- Nawan Shahar (Mandi Aur) → Shahid Bhagat Singh Nagar
- Raikot (Talwandi Rai) → Ludhiana
- Sunam → Sangrur
- Talwandi Sabo (Shekhupura) → Bathinda

Production comparison:

- 92 Rice production keys
- 89 with price data
- 3 without price data
- 0 extra price-only rows

The three production keys without Rice prices were Pathankot combinations in later crop years.


# 42. Punjab Price Stage Complete

At this point all Punjab crops were complete:

- Wheat
- Maize
- Rice

The final reusable market mapping was saved as:

`punjab_market_district_mapping.csv`

A Punjab price-coverage summary was also saved for later quality review.


# 43. Haryana Official Market Reference

Haryana required its own market-to-district mapping.

The Haryana State Agricultural Marketing Board (HSAMB) district-wise market table was used as the official reference.

The source table contained district relationships for:

- Principal Yards / Market Committees
- Purchase Centers
- Sub Yards

After reshaping and de-duplicating the table, the official reference contained hundreds of market-location relationships.

District names were standardized to match the production dataset.

Examples:

`Mohindergarh` → `Mahendragarh`

`Yamuna Nagar` → `Yamunanagar`

The processed official reference was saved as:

`haryana_official_market_reference.csv`


In [18]:
# Haryana helper functions used when market names contain extra text
# or punctuation.

def extract_parent_market(name):
    """Return the main market name before any parenthesized detail."""
    name = str(name)
    name = re.sub(
        r"\bAPMC\b",
        "",
        name,
        flags=re.IGNORECASE
    )
    return name.split("(")[0].strip()


def compact_market_key(name):
    """Create an aggressive compact comparison key for fuzzy/manual review."""
    name = str(name).lower().strip()

    # Replace punctuation with spaces.
    name = re.sub(
        r"[^a-z0-9\s]",
        " ",
        name
    )

    # Remove the source label.
    name = re.sub(
        r"\bapmc\b",
        " ",
        name
    )

    # Remove all remaining whitespace for comparison.
    return re.sub(
        r"\s+",
        "",
        name
    )


# 44. Haryana Wheat — Selenium Failure and Recovery

Haryana Wheat was the most useful Selenium debugging case.

## First result

- 38 months discovered as available
- 16 months unavailable
- 0 successful downloads
- 38 technical failures

Because month discovery and the report page worked, the problem was traced to the export selector rather than source availability.

The exact `Export as CSV` selector fixed this.

A later recovery version then produced:

`javascript error: Invalid or unexpected token`

because the JavaScript clickable-parent selector was passed incorrectly.

After rewriting the JavaScript expression on one line, downloads succeeded normally.


# 45. Haryana Wheat — Market Mapping

Haryana Wheat contained **104 unique market names**.

Mapping was built through:

1. direct district-name matches,
2. official HSAMB exact matches,
3. parent-market exact matches,
4. reviewed high-confidence fuzzy candidates,
5. manual verification of remaining cases.

An important ambiguity was `Dhand`, which appeared in more than one official relationship.

For `Dhand APMC`, the principal-market interpretation was selected:

`Dhand` → `Kaithal`

A dangerous fuzzy match also demonstrated why automated fuzzy results must be reviewed:

`New Grain Market(main), Karnal APMC`

was incorrectly suggested as a Jhajjar/Beri match.

It was manually corrected to:

`Karnal`.


# 46. Haryana Wheat — Final Results

Verified monthly history:

- **38 files**
- **910 cleaned rows**
- **104 markets**
- first month: March 2021
- last month: June 2025

Final Haryana master mapping at this stage:

- 104 unique market keys
- 0 mapping conflicts
- 0 unmapped rows
- 0 invalid production districts

Annual Wheat features:

- **76 feature rows**
- **0 duplicate keys**

Coverage:

- Low: 73
- Moderate: 2
- Good: 1

Production comparison:

- 88 production keys
- 76 with price data
- 12 without price data
- 0 extra price-only rows


# 47. Haryana Maize — Final Results

After the Selenium export fix, Haryana Maize collection completed successfully.

Collection summary:

- 40 files downloaded in the final run
- 1 existing file skipped
- 13 months unavailable
- 0 failures
- 41 Maize files on disk

Loaded history:

- **255 rows**
- **41 available months**
- **23 markets**
- first month: March 2021
- last month: June 2025

Two new markets required mapping:

- Bahadurgarh → Jhajjar
- Iamailabad → Kurukshetra

`Iamailabad` was treated as an AGMARKNET spelling variation of Ismailabad.


# 48. Haryana Maize — Annual Features

Final Haryana Maize annual price table:

- **30 feature rows**
- **0 duplicate keys**

Coverage:

- Low: 24
- Moderate: 5
- Complete: 1

Missing volatility:

- 24 rows

Production comparison:

- 70 production keys
- 27 with price data
- 43 without price data
- 3 price rows not present in production

This sparse coverage is a source-data limitation and is deliberately preserved.


# 49. Haryana Rice — Cascading Dropdown Timing Fix

Haryana Rice exposed a different Selenium issue.

During the first run, errors included:

`Select Commodity dropdown is disabled`

and:

`Year dropdown is disabled`

The script was interacting with dependent dropdowns before React had finished enabling them.

The final dropdown helper therefore waits until the control's `tabindex` is no longer `-1`.

After applying the wait-based helper, the failed October 2021 test succeeded and the adaptive Rice collector was safely rerun.

Already downloaded files were skipped automatically.


# 50. Haryana Rice — Final Collection

Haryana Rice used the AGMARKNET `Paddy(Common)` alias.

Final collection:

- 29 files downloaded during the successful rerun
- 1 file skipped because it already existed
- 24 months unavailable
- 0 failed downloads
- 30 Rice files on disk

Loaded history:

- **492 cleaned rows**
- **30 available months**
- **59 markets**
- first month: October 2021
- last month: April 2025


# 51. Haryana Rice — Mapping Updates

The existing Haryana master mapping resolved almost all Rice markets.

Three additional mappings were required:

- Beri → Jhajjar
- Bhiwani → Bhiwani
- Gohana → Sonipat

After adding these mappings:

- 0 rows had missing district
- 0 markets remained unmapped

The updated Haryana master mapping was saved and reused for final Rice feature generation.


# 52. Haryana Rice — Annual Features

Final Haryana Rice annual feature table:

- **45 feature rows**

Coverage:

- Low: 39
- Moderate: 6

Missing volatility:

- 39 rows

Production comparison:

- 86 production keys
- 45 with price data
- 51 without price data
- 0 extra price-only rows

The low coverage is retained transparently rather than hidden through artificial imputation.


# 53. Price Processing Completed for All 9 State-Crop Combinations

The price stage was completed for every required combination.

| State | Wheat | Rice | Maize |
|---|---|---|---|
| Uttar Pradesh | Complete | Complete | Complete |
| Punjab | Complete | Complete | Complete |
| Haryana | Complete | Complete | Complete |

Each state-crop output uses the same annual schema, which makes the nine tables directly concatenable.


# 54. Combine All Annual Price Feature Files

The nine state-crop feature files are combined into one master price table.

Expected file count:

`3 states × 3 crops = 9`

Before saving the master table, the merge-key columns are standardized:

- State
- District
- Crop
- Year

The final key must remain unique:

`State + District + Crop + Year`

A duplicate at this stage could multiply production rows during later integration, so duplicate keys are treated as a critical validation failure.


In [19]:
# Locate the nine state-crop annual price-feature files.
price_feature_files = sorted(
    PROCESSED_DIR.glob(
        "*_price_features_2021_2025.csv"
    )
)

print(
    "Price feature files found:",
    len(price_feature_files)
)

EXPECTED_PRICE_FEATURE_FILES = 9

PRICE_FEATURES_READY = (
    len(price_feature_files)
    == EXPECTED_PRICE_FEATURE_FILES
)

if not PRICE_FEATURES_READY:
    print(
        "Expected exactly 9 state-crop feature files "
        "(3 states × 3 crops)."
    )
    print(
        "The final master-price rebuild will be skipped instead "
        "of crashing the notebook."
    )

else:
    all_price_features = []

    for file in price_feature_files:
        temp = pd.read_csv(file)

        print(
            file.name,
            "→",
            len(temp),
            "rows"
        )

        all_price_features.append(
            temp
        )

    price_features_master = pd.concat(
        all_price_features,
        ignore_index=True
    )

    # Standardize final merge keys.
    for column in [
        "State",
        "District",
        "Crop",
        "Year"
    ]:
        price_features_master[column] = (
            price_features_master[
                column
            ]
            .astype(str)
            .str.strip()
        )

    duplicate_master_keys = (
        price_features_master
        .duplicated(
            subset=[
                "State",
                "District",
                "Crop",
                "Year"
            ]
        )
        .sum()
    )

    print(
        "Combined price rows:",
        len(price_features_master)
    )

    print(
        "Duplicate master keys:",
        duplicate_master_keys
    )

    if duplicate_master_keys != 0:
        raise ValueError(
            "Duplicate State-District-Crop-Year keys were found "
            "in the combined price features."
        )

Price feature files found: 9
haryana_maize_price_features_2021_2025.csv → 30 rows
haryana_rice_price_features_2021_2025.csv → 45 rows
haryana_wheat_price_features_2021_2025.csv → 76 rows
punjab_maize_price_features_2021_2025.csv → 57 rows
punjab_rice_price_features_2021_2025.csv → 89 rows
punjab_wheat_price_features_2021_2025.csv → 87 rows
up_maize_price_features_2021_2025.csv → 163 rows
up_rice_price_features_2021_2025.csv → 263 rows
up_wheat_price_features_2021_2025.csv → 294 rows
Combined price rows: 1104
Duplicate master keys: 0


# 55. Discovery of Zero-Month Placeholder Rows

The first combined master table contained:

- **1,104 rows**
- **45 rows with missing `Avg_Modal_Price`**

Inspection showed that every one of those 45 rows also had:

`Months_Available = 0`

Validation confirmed:

- Missing Avg Price: 45
- Rows with 0 months: 45
- Missing price but months > 0: 0
- Price exists but months == 0: 0

Therefore these were not genuine price-feature observations.

They were placeholder State-District-Crop-Year rows introduced during earlier production-coverage auditing.

A standalone price table should contain only combinations supported by actual AGMARKNET observations.

The placeholder rows were therefore removed from the master **price** dataset.

They are not lost from the final project: when the cleaned price table is later left-joined onto production, those combinations correctly receive `NaN` price values.


In [20]:
if not globals().get(
    "PRICE_FEATURES_READY",
    False
):
    print(
        "Skipped master-price cleaning because the 9 required "
        "state-crop feature files were not all available."
    )

else:
    # Keep only rows supported by at least one real monthly price
    # observation.
    #
    # Missing production-price combinations are NOT filled with zero.
    # They reappear naturally as NaN after the later left join.
    price_features_master_clean = (
        price_features_master[
            price_features_master[
                "Months_Available"
            ] > 0
        ]
        .copy()
        .reset_index(drop=True)
    )

    print(
        "Rows before:",
        len(price_features_master)
    )

    print(
        "Rows after:",
        len(price_features_master_clean)
    )

    print(
        "Removed placeholder rows:",
        len(price_features_master)
        - len(price_features_master_clean)
    )

    print(
        "Missing average price:",
        price_features_master_clean[
            "Avg_Modal_Price"
        ].isna().sum()
    )

    print(
        "Duplicate keys:",
        price_features_master_clean
        .duplicated(
            subset=[
                "State",
                "District",
                "Crop",
                "Year"
            ]
        )
        .sum()
    )

    price_features_master = (
        price_features_master_clean.copy()
    )

Rows before: 1104
Rows after: 1059
Removed placeholder rows: 45
Missing average price: 0
Duplicate keys: 0


# 56. Final Master Price Dataset

After removing zero-month placeholders:

- rows before cleaning: **1,104**
- placeholder rows removed: **45**
- final real price-feature rows: **1,059**

Final validation:

- missing `Avg_Modal_Price`: **0**
- duplicate final keys: **0**
- zero/negative average prices: **0**

Missing `Price_Volatility_Pct` is still allowed because volatility is intentionally left missing when fewer than six months are available.

The final dataset is saved as:

`data/processed/price_features_2021_2025.csv`


In [21]:
if not globals().get(
    "PRICE_FEATURES_READY",
    False
):
    print(
        "Skipped final price-master save because the required "
        "nine state-crop feature files were not all available."
    )

else:
    PROCESSED_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    final_price_path = (
        PROCESSED_DIR
        / "price_features_2021_2025.csv"
    )

    price_features_master.to_csv(
        final_price_path,
        index=False
    )

    print(
        "MASTER PRICE DATASET SAVED ✅"
    )

    print(
        "Saved at:",
        final_price_path.resolve()
    )

    print(
        "Rows:",
        len(price_features_master)
    )

MASTER PRICE DATASET SAVED ✅
Saved at: E:\Crop-Yeild-Risk-Dashboard\data\processed\price_features_2021_2025.csv
Rows: 1059


# 57. Final Validation Rules

Before the price table is considered complete, verify all of the following.

## Grain

Exactly one row per:

`State + District + Crop + Year`

## Average Price

- no negative values,
- no zero values,
- no missing values inside the standalone cleaned price table.

## Volatility

Missing volatility is acceptable when:

`Months_Available < 6`

It must not be replaced with zero.

## Coverage

Retain:

- `Months_Available`
- `Price_Coverage_Pct`
- `Price_Coverage_Quality`

These fields communicate how strongly each annual price feature is supported by the source.

## Mapping

No unresolved market is silently assigned to a district.

## Missing Source Coverage

A production row without AGMARKNET price coverage will receive `NaN` only after the cleaned price table is left-joined onto production.


# 58. Complete Price Methodology Flow

```text
AGMARKNET monthly analysis page
        ↓
Select Commodity + State + Year
        ↓
Wait for cascading controls to enable
        ↓
Discover months actually available
        ↓
Download available monthly CSV reports
        ↓
Read each file using header=1
        ↓
Keep Market + selected-month price
        ↓
Remove footer/non-APMC rows
        ↓
Normalize Market_Key
        ↓
Map Market → District
        ↓
Use official references where possible
        ↓
Use geocoding / reviewed matching for unresolved markets
        ↓
Validate district names against production
        ↓
Convert calendar month → crop year
        ↓
Median across markets within each district-month
        ↓
Aggregate monthly district prices to annual features
        ↓
Calculate volatility and coverage
        ↓
Set volatility = NA when Months_Available < 6
        ↓
Validate unique State-District-Crop-Year grain
        ↓
Repeat for 3 states × 3 crops
        ↓
Combine all 9 annual feature tables
        ↓
Identify and remove zero-month placeholder rows
        ↓
Final clean price master
        ↓
price_features_2021_2025.csv
```


# 59. Consolidated Failure Log and Engineering Lessons

This notebook intentionally keeps the failed approaches because they explain why the final pipeline is designed the way it is.

## Failure 1 — Wrong CSV header

AGMARKNET places a report title in the first row.

**Fix:** `pd.read_csv(..., header=1)`.

---

## Failure 2 — Footer rows treated as markets

Average/Note/report rows entered the dataset.

**Fix:** keep only market rows ending with `APMC`.

---

## Failure 3 — Weak direct annual aggregation

Aggregating every market row directly could overweight districts with many markets and mix between-market differences with temporal volatility.

**Fix:** first calculate district-month median price, then calculate annual features.

---

## Failure 4 — Assuming all 12 months exist

Punjab month selection timed out.

**Investigation:** the dropdown was enabled, but the requested month was not actually offered by the source.

**Fix:** discover available months before downloading.

---

## Failure 5 — Rice terminology mismatch

Project crop = `Rice`; suitable source commodity = `Paddy(Common)`.

**Fix:** explicit source-to-project alias.

---

## Failure 6 — Lost Jupyter variables

Slow geocoding outputs disappeared from active memory after kernel/session changes.

**Fix:** save expensive intermediate results to disk and reload them.

---

## Failure 7 — `No objects to concatenate`

`pd.concat()` was called on an empty list.

**Fix:** check list length before concatenation.

---

## Failure 8 — Helper function not defined

`load_price_history` disappeared from active kernel state.

**Fix:** rerun helper-function definitions and reload existing raw files.

---

## Failure 9 — Haryana Wheat export selector

All available months failed because a broad XPath matched ancestor elements rather than the CSV action.

**Fix:** exact text selector plus clickable-parent lookup.

---

## Failure 10 — Invalid JavaScript token

A multi-line JavaScript selector string was malformed.

**Fix:** valid one-line `closest(...)` expression.

---

## Failure 11 — Cascading dropdown disabled

Haryana Rice attempted to use Commodity/Year controls before React had enabled them.

**Fix:** explicitly wait until `tabindex != -1`.

---

## Failure 12 — Missing average price in combined master

45 rows had missing average price.

**Investigation:** all 45 also had zero available months.

**Conclusion:** placeholder coverage rows, not real price observations.

**Fix:** remove zero-month rows from the standalone price master. Allow price to become `NaN` naturally after left-joining onto production.

---

## Main Engineering Lessons

1. Source absence and automation failure are different conditions.
2. Missing values must not automatically become zero.
3. Geographic mapping requires canonical district validation.
4. Fuzzy matching should suggest candidates, not silently decide locations.
5. Expensive intermediate work should be saved to disk.
6. Coverage information should travel with each price feature.
7. Final grain uniqueness must be validated before integration.


# 60. Important Output Files

## Uttar Pradesh

- `up_market_district_mapping.csv`
- `up_wheat_monthly_prices_mapped_2021_2025.csv`
- `up_wheat_price_features_2021_2025.csv`
- `up_maize_monthly_prices_mapped_2021_2025.csv`
- `up_maize_price_features_2021_2025.csv`
- `up_rice_monthly_prices_mapped_2021_2025.csv`
- `up_rice_price_features_2021_2025.csv`

## Punjab

- `punjab_market_district_mapping.csv`
- Punjab Wheat/Maize/Rice mapped monthly outputs
- Punjab Wheat/Maize/Rice annual price-feature outputs
- `punjab_price_coverage_summary.csv`

## Haryana

- `haryana_official_market_reference.csv`
- `haryana_market_district_mapping.csv`
- `haryana_wheat_monthly_prices_mapped_2021_2025.csv`
- `haryana_wheat_price_features_2021_2025.csv`
- `haryana_maize_monthly_prices_mapped_2021_2025.csv`
- `haryana_maize_price_features_2021_2025.csv`
- `haryana_rice_monthly_prices_mapped_2021_2025.csv`
- `haryana_rice_price_features_2021_2025.csv`

## Final Combined Output

- `price_features_2021_2025.csv`


# Run-All Safety Summary

The corrected notebook is designed so that historical development artifacts do
not stop a normal rerun.

### Skipped by default

- live Selenium diagnostics,
- historical Punjab geocoding,
- recovery from `punjab_unmapped_market_geocode_candidates.csv`,
- temporary `punjab_mapping_parts`,
- temporary `punjab_geocoded_found`,
- temporary `punjab_market_mapping_v2`,
- old mismatch-review objects.

### Used by the completed pipeline

- downloaded raw price CSVs when available,
- final market-mapping files,
- nine annual state-crop price-feature files,
- final master output `price_features_2021_2025.csv`.

If a final reusable file is missing, the notebook reports the missing path
clearly instead of trying to recreate an expensive historical step
automatically.

# 61. Conclusion

The price-data stage is complete for:

- Uttar Pradesh — Wheat, Rice, Maize
- Punjab — Wheat, Rice, Maize
- Haryana — Wheat, Rice, Maize

The final output contains district-crop-year price features generated from the AGMARKNET monthly observations that were actually available.

The pipeline explicitly preserves source limitations rather than hiding them through fabricated values.

Final price master:

`price_features_2021_2025.csv`

The next analytical stage is:

```text
Production
    +
Rainfall
    +
Price
    ↓
Combined analytical dataset
```

The clean price master should be merged using a **left join from the production/rainfall base**, so districts without AGMARKNET price coverage remain in the project with price fields set to missing rather than being dropped.
